# Breathprint Edge - DistilGPT2 Optimisation Benchmark
**Goal:** Demonstrate INT8 dynamic quantisation, ONNX Runtime, structured attention-head pruning, and early-exit inference on a DistilGPT2 backbone adapted for UCI Gas Sensor VOC classification.

**Run cells top-to-bottom. All state flows from one cell to the next.**

**Important architecture change:** GPT-style causal attention cannot use a front `[CLS]` token because it cannot attend to future sensor tokens. This notebook appends a learned **readout token at the end** of the 128 sensor-token sequence and classifies using that final token.


## 1. Install dependencies


In [1]:
%pip install -q transformers ucimlrepo onnxruntime
import torch, transformers
print('PyTorch     :', torch.__version__)
print('Transformers:', transformers.__version__)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 49.2 MB/s eta 0:00:00
PyTorch     : 2.10.0+cu128
Transformers: 5.0.0


## 2. Imports and Data


In [2]:
from ucimlrepo import fetch_ucirepo
import pandas as pd, numpy as np, copy, os, time, tracemalloc, tempfile, zipfile, warnings
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score
import torch, torch.nn as nn, torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from transformers import GPT2Model
warnings.filterwarnings('ignore')

# ── Fetch UCI Gas Sensor dataset (id=270) ─────────────────────────────────
gas   = fetch_ucirepo(id=270)
X_raw = gas.data.features
y_raw = gas.data.targets

rows = []
for idx_str, row in X_raw.iterrows():
    class_id, concentration = idx_str.split(';')
    fv = {}
    for cell in row:
        if pd.isna(cell): continue
        i, v = str(cell).strip().split(':')
        fv[int(i)] = float(v)
    rows.append({'class': int(class_id), 'concentration': float(concentration), **fv})

df = pd.DataFrame(rows)
df[1] = [float(str(c).strip().split(':')[1]) for c in y_raw.iloc[:, 0]]
feature_cols = sorted(c for c in df.columns if isinstance(c, int))
X = df[feature_cols].astype(float)
X.columns = [f'Feature{i}' for i in range(1, 129)]
y = df['class'].astype(int) - 1

X_tr, X_tmp, y_tr, y_tmp = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y)
X_val, X_test, y_val, y_test = train_test_split(
    X_tmp, y_tmp, test_size=0.50, random_state=42, stratify=y_tmp)

scaler = StandardScaler()
X_tr   = scaler.fit_transform(X_tr)
X_val  = scaler.transform(X_val)
X_test = scaler.transform(X_test)

to_t = lambda a: torch.tensor(a, dtype=torch.float32)
X_tr_t, X_val_t, X_test_t = to_t(X_tr), to_t(X_val), to_t(X_test)
y_tr_t  = torch.tensor(y_tr.values,  dtype=torch.long)
y_val_t = torch.tensor(y_val.values, dtype=torch.long)
y_tst_t = torch.tensor(y_test.values,dtype=torch.long)

BATCH = 64
train_loader = DataLoader(TensorDataset(X_tr_t,  y_tr_t),  batch_size=BATCH, shuffle=True)
val_loader   = DataLoader(TensorDataset(X_val_t, y_val_t), batch_size=BATCH)
test_loader  = DataLoader(TensorDataset(X_test_t,y_tst_t), batch_size=BATCH)

CLASS_MAP = {0:'Ethanol',1:'Ethylene',2:'Ammonia',
             3:'Acetaldehyde',4:'Acetone',5:'Toluene'}
print(f'Train {len(X_tr_t)} | Val {len(X_val_t)} | Test {len(X_test_t)}')
print(f'Features: 128  Classes: 6')


Train 9737 | Val 2086 | Test 2087
Features: 128  Classes: 6


## 3. Model Definition

Key design decisions:
- `_embed()` projects each scalar sensor reading to a 768-dim token
- A learned **readout token is appended at the end** of the sequence
- DistilGPT2 receives `inputs_embeds`-style continuous sensor embeddings, not text token IDs
- `_run_transformer()` manually steps through all 6 GPT blocks so early exits and final inference share the same hidden-state path
- Exit heads classify from the readout token at position `-1`


In [3]:
class DistilGPTSensorClassifier(nn.Module):
    N_LAYERS = 6
    D_MODEL  = 768

    def __init__(self, num_features=128, num_classes=6, dropout=0.1):
        super().__init__()
        D = self.D_MODEL
        self.num_features = num_features

        # Sensor scalar -> GPT token embedding
        self.value_proj = nn.Linear(1, D)

        # GPT is causal, so the readout token must be at the END.
        # The final readout token can attend to all previous sensor tokens.
        self.readout_token = nn.Parameter(torch.randn(1, 1, D) * 0.02)

        # DistilGPT2 backbone: 6 layers, 768 hidden size, 12 attention heads
        self.distilgpt = GPT2Model.from_pretrained("distilgpt2")

        # We are not using text token IDs, but keep the pretrained positional
        # embeddings and transformer blocks. Replace vocab embedding with Identity
        # so it is not saved as part of the sensor model state_dict.
        self.distilgpt.wte = nn.Identity()

        # Final classifier head
        self.pre_classifier = nn.Linear(D, D)
        self.classifier     = nn.Linear(D, num_classes)
        self.dropout        = nn.Dropout(dropout)

        # One lightweight exit head per GPT block
        self.exit_heads = nn.ModuleList([
            nn.Sequential(
                nn.LayerNorm(D),
                nn.Linear(D, 256),
                nn.GELU(),
                nn.Dropout(dropout),
                nn.Linear(256, num_classes),
            )
            for _ in range(self.N_LAYERS)
        ])

    def _embed(self, x):
        """
        x: [B, 128]
        returns: [B, 129, 768]
        sequence = 128 sensor tokens + 1 final readout token
        """
        x = self.value_proj(x.unsqueeze(-1))  # [B, 128, 768]
        readout = self.readout_token.expand(x.size(0), -1, -1)
        hidden = torch.cat([x, readout], dim=1)  # [B, 129, 768]

        # Add GPT2 positional embeddings manually because we pass embeddings,
        # not token IDs.
        seq_len = hidden.size(1)
        position_ids = torch.arange(seq_len, device=hidden.device).unsqueeze(0)
        hidden = hidden + self.distilgpt.wpe(position_ids)
        hidden = self.distilgpt.drop(hidden)
        return hidden

    def _run_transformer(self, emb):
        """Manual GPT block pass so we can collect hidden states for exits."""
        hidden = emb
        layer_hiddens = []

        for block in self.distilgpt.h:
            out = block(
                hidden,
                layer_past=None,
                attention_mask=None,
                head_mask=None,
                use_cache=False,
                output_attentions=False,
            )
            hidden = out[0]
            layer_hiddens.append(hidden)

        hidden = self.distilgpt.ln_f(hidden)
        return hidden, layer_hiddens

    def freeze_backbone(self):
        for p in self.distilgpt.parameters():
            p.requires_grad = False

    def unfreeze_backbone(self):
        for p in self.distilgpt.parameters():
            p.requires_grad = True

    def forward(self, x, return_all_exits=False):
        emb = self._embed(x)
        hidden, layer_hiddens = self._run_transformer(emb)

        # Use final readout token representation
        readout_hidden = hidden[:, -1]

        cls_out = self.dropout(F.relu(self.pre_classifier(readout_hidden)))
        final = self.classifier(cls_out)

        if not return_all_exits:
            return final

        exits = [
            self.exit_heads[i](h[:, -1])
            for i, h in enumerate(layer_hiddens)
        ]

        return exits, final


# Backward-compatible alias so existing training/evaluation cells keep working.
DistilBertSensorClassifier = DistilGPTSensorClassifier

print("DistilGPT2 sensor classifier defined.")


DistilGPT2 sensor classifier defined.


## 4. Training

**Phase 1 (5 epochs):** Backbone frozen. Only input adapter + classification head trained.

**Phase 2 (up to 15 epochs):** Full fine-tune with differential LRs.
Exit heads co-trained with weighted loss (60% final + 40% intermediate average).


In [5]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

model = DistilGPTSensorClassifier().to(device)
total = sum(p.numel() for p in model.parameters())
print(f'Total params: {total:,}  ({total/1e6:.1f} M)')

criterion = nn.CrossEntropyLoss(label_smoothing=0.05)

@torch.no_grad()
def evaluate(m, loader):
    m.eval()
    y_true, y_pred = [], []
    for xb, yb in loader:
        y_pred.extend(m(xb.to(device)).argmax(1).cpu().tolist())
        y_true.extend(yb.tolist())
    return accuracy_score(y_true, y_pred)

def run_epoch(m, loader, opt, use_exit_loss=False):
    m.train()
    total_loss = 0.0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        opt.zero_grad()
        if use_exit_loss:
            exits, final = m(xb, return_all_exits=True)
            loss = (0.6 * criterion(final, yb) +
                    0.4 * sum(criterion(e, yb) for e in exits) / len(exits))
        else:
            loss = criterion(m(xb), yb)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(m.parameters(), 1.0)
        opt.step()
        total_loss += loss.item()
    return total_loss / len(loader)

# ── PHASE 1: frozen backbone ──────────────────────────────────────────────
model.freeze_backbone()
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'\n-- Phase 1: {trainable:,} trainable params (backbone frozen) --')

opt1   = optim.AdamW([p for p in model.parameters() if p.requires_grad],
                     lr=3e-4, weight_decay=1e-4)
sched1 = optim.lr_scheduler.ReduceLROnPlateau(opt1, mode='max', factor=0.5, patience=2)
best_state, best_val = None, 0.0

for ep in range(5):
    loss    = run_epoch(model, train_loader, opt1)
    val_acc = evaluate(model, val_loader)
    sched1.step(val_acc)
    print(f'  Epoch {ep+1:02d} | loss={loss:.4f} | val_acc={val_acc:.4f}')
    if val_acc > best_val:
        best_val, best_state = val_acc, copy.deepcopy(model.state_dict())
model.load_state_dict(best_state)
print(f'  Best: {best_val:.4f}')

# ── PHASE 2: full fine-tune ───────────────────────────────────────────────
# ── PHASE 2: full fine-tune ───────────────────────────────────────────────
model.unfreeze_backbone()
total2 = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'\n-- Phase 2: {total2:,} trainable params --')

opt2 = optim.AdamW([
    {'params': model.distilgpt.parameters(),       'lr': 2e-5},
    {'params': model.value_proj.parameters(),      'lr': 1e-4},
    {'params': [model.readout_token],              'lr': 1e-4},
    {'params': model.pre_classifier.parameters(),  'lr': 1e-4},
    {'params': model.classifier.parameters(),      'lr': 1e-4},
    {'params': model.exit_heads.parameters(),      'lr': 1e-4},
], weight_decay=1e-4)

sched2 = optim.lr_scheduler.ReduceLROnPlateau(
    opt2, mode='max', factor=0.5, patience=3
)

patience_cnt = 0

for ep in range(15):
    loss = run_epoch(model, train_loader, opt2, use_exit_loss=True)
    val_acc = evaluate(model, val_loader)
    sched2.step(val_acc)

    print(f'  Epoch {ep+1:02d} | loss={loss:.4f} | val_acc={val_acc:.4f}')

    if val_acc > best_val:
        best_val = val_acc
        best_state = copy.deepcopy(model.state_dict())
        patience_cnt = 0
    else:
        patience_cnt += 1
        if patience_cnt >= 7:
            print('  Early stopping.')
            break

model.load_state_dict(best_state)
test_acc = evaluate(model, test_loader)

print(f'\nBest val : {best_val:.4f}  |  Test acc : {test_acc:.4f}')

torch.save(model.state_dict(), 'distilgpt_sensor_best.pt')
print('Saved -> distilgpt_sensor_best.pt')

Device: cuda


Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

GPT2Model LOAD REPORT from: distilgpt2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
transformer.h.{0, 1, 2, 3, 4, 5}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Total params: 45,112,362  (45.1 M)

-- Phase 1: 1,797,162 trainable params (backbone frozen) --
  Epoch 01 | loss=1.5822 | val_acc=0.5101
  Epoch 02 | loss=1.4298 | val_acc=0.5158
  Epoch 03 | loss=1.3353 | val_acc=0.5197
  Epoch 04 | loss=1.2911 | val_acc=0.5642
  Epoch 05 | loss=1.2549 | val_acc=0.5316
  Best: 0.5642

-- Phase 2: 45,112,362 trainable params --
  Epoch 01 | loss=1.3199 | val_acc=0.6246
  Epoch 02 | loss=1.1248 | val_acc=0.7105
  Epoch 03 | loss=0.9924 | val_acc=0.7430
  Epoch 04 | loss=0.8964 | val_acc=0.8015
  Epoch 05 | loss=0.8189 | val_acc=0.8279
  Epoch 06 | loss=0.7593 | val_acc=0.8538
  Epoch 07 | loss=0.7084 | val_acc=0.8778
  Epoch 08 | loss=0.6723 | val_acc=0.8797
  Epoch 09 | loss=0.6356 | val_acc=0.9032
  Epoch 10 | loss=0.6111 | val_acc=0.8888
  Epoch 11 | loss=0.5834 | val_acc=0.9228
  Epoch 12 | loss=0.5628 | val_acc=0.9334
  Epoch 13 | loss=0.5497 | val_acc=0.9286
  Epoch 14 | loss=0.5273 | val_acc=0.9310
  Epoch 15 | loss=0.5179 | val_acc=0.9367

Best

## 5. Benchmark Utilities


In [6]:
# Shared by all 5 stages.
# n_warmup=5, n_runs=50 keeps CPU benchmark time reasonable on Colab.
results_table = {}

def measure_latency_ms(fwd, x, n_warmup=5, n_runs=50):
    for _ in range(n_warmup):
        with torch.no_grad(): fwd(x)
    times = []
    for _ in range(n_runs):
        t0 = time.perf_counter()
        with torch.no_grad(): fwd(x)
        times.append((time.perf_counter() - t0) * 1000)
    return float(np.mean(times)), float(np.percentile(times, 95))

def measure_peak_ram_mb(fwd, x):
    tracemalloc.start()
    with torch.no_grad(): fwd(x)
    _, peak = tracemalloc.get_traced_memory()
    tracemalloc.stop()
    return peak / 1024**2

def model_disk_mb(m):
    with tempfile.NamedTemporaryFile(suffix='.pt', delete=False) as f: path = f.name
    torch.save(m.state_dict(), path)
    mb = os.path.getsize(path) / 1024**2
    os.unlink(path); return mb

def file_mb(path): return os.path.getsize(path) / 1024**2

@torch.no_grad()
def eval_acc(fwd, loader):
    y_true, y_pred = [], []
    for xb, yb in loader:
        out = fwd(xb)
        preds = (np.argmax(out, 1).tolist() if isinstance(out, np.ndarray)
                 else out.argmax(1).tolist())
        y_pred.extend(preds); y_true.extend(yb.tolist())
    return accuracy_score(y_true, y_pred)

def record_stage(name, fwd, loader, disk_mb_val):
    lat_mean, lat_p95 = measure_latency_ms(fwd, X_test_t[:1])
    ram_mb            = measure_peak_ram_mb(fwd, X_test_t[:1])
    acc               = eval_acc(fwd, loader)
    results_table[name] = {
        'Accuracy':       round(acc,        4),
        'Lat Mean (ms)':  round(lat_mean,   1),
        'Lat p95  (ms)':  round(lat_p95,    1),
        'Peak RAM (MB)':  round(ram_mb,     2),
        'Disk Size (MB)': round(disk_mb_val,2),
    }
    print(f'\n[{name}]')
    print(f'  Accuracy  : {acc:.4f}')
    print(f'  Latency   : {lat_mean:.1f} ms  (p95 = {lat_p95:.1f} ms)')
    print(f'  Peak RAM  : {ram_mb:.2f} MB')
    print(f'  Disk size : {disk_mb_val:.2f} MB')

print('Benchmark utilities ready.')

Benchmark utilities ready.


## Stage 1 — Baseline FP32


In [7]:
model_fp32 = copy.deepcopy(model).cpu().eval()
record_stage('1 - Baseline (FP32)', model_fp32, test_loader,
             disk_mb_val=model_disk_mb(model_fp32))


[1 - Baseline (FP32)]
  Accuracy  : 0.9353
  Latency   : 170.8 ms  (p95 = 211.2 ms)
  Peak RAM  : 0.01 MB
  Disk size : 172.13 MB


## Stage 2 — INT8 Dynamic Post-Training Quantisation

This applies PyTorch dynamic quantisation to `nn.Linear` modules in the sensor adapter, classifier, and exit heads. Hugging Face GPT2 attention/MLP projections use custom `Conv1D` layers, so this cell keeps the same PTQ technique but does not quantise every transformer projection the way a pure-`nn.Linear` architecture would.


In [8]:
model_int8 = torch.quantization.quantize_dynamic(
    copy.deepcopy(model).cpu().eval(),
    {nn.Linear}, dtype=torch.qint8,
)
record_stage('2 - INT8 PTQ', model_int8, test_loader,
             disk_mb_val=model_disk_mb(model_int8))


[2 - INT8 PTQ]
  Accuracy  : 0.9305
  Latency   : 169.7 ms  (p95 = 215.2 ms)
  Peak RAM  : 0.01 MB
  Disk size : 167.04 MB


In [10]:
!pip install onnx
!pip install onnxruntime

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.6/17.6 MB 57.6 MB/s eta 0:00:00


## Stage 3 — ONNX Runtime

Exported with `dynamo=False` (legacy TorchScript exporter) and `opset_version=18`
to avoid the `aten.sym_stride` dispatch error in newer PyTorch dynamo exporters.
ORT session uses 4 threads matching the Raspberry Pi 5 quad-core Cortex-A76.


In [11]:
import onnxruntime as ort
ONNX_PATH = 'distilgpt_sensor.onnx'

# Wrap to avoid any early-exit branch in the trace
class _ONNXWrapper(nn.Module):
    def __init__(self, m): super().__init__(); self.m = m
    def forward(self, x):  return self.m(x)

onnx_src = _ONNXWrapper(copy.deepcopy(model).cpu().eval())

torch.onnx.export(
    onnx_src, X_test_t[:1], ONNX_PATH,
    dynamo=False, opset_version=18,
    input_names=['input'], output_names=['logits'],
    dynamic_axes={'input': {0: 'batch'}, 'logits': {0: 'batch'}},
    verbose=False,
)
print(f'Exported -> {ONNX_PATH}  ({file_mb(ONNX_PATH):.2f} MB)')

so = ort.SessionOptions()
so.inter_op_num_threads = 4
so.intra_op_num_threads = 4
ort_sess = ort.InferenceSession(
    ONNX_PATH, sess_options=so, providers=['CPUExecutionProvider'])

def ort_fwd(x): return ort_sess.run(None, {'input': x.detach().numpy()})[0]

record_stage('3 - ONNX Runtime', ort_fwd, test_loader,
             disk_mb_val=file_mb(ONNX_PATH))


Exported -> distilgpt_sensor.onnx  (167.67 MB)

[3 - ONNX Runtime]
  Accuracy  : 0.9353
  Latency   : 477.8 ms  (p95 = 797.3 ms)
  Peak RAM  : 0.00 MB
  Disk size : 167.67 MB


## Stage 4 — Structured Head Pruning (33% attention heads physically removed)

DistilGPT2 exposes built-in `prune_heads()` support for GPT2 attention modules. Heads are selected by lowest norm of their V-projection slice from the combined QKV projection (`c_attn`).

This physically removes attention-head parameters and then fine-tunes for recovery.


In [14]:
model_pruned = copy.deepcopy(model).cpu()

def get_gpt_heads_to_prune(m, n_prune=4):
    """
    Select lowest-norm attention heads from each DistilGPT2 layer.
    GPT2 attention uses combined QKV projection: c_attn.
    """
    result = {}

    for layer_idx, block in enumerate(m.distilgpt.h):
        attn = block.attn
        n_heads = attn.num_heads
        head_dim = attn.head_dim
        D = n_heads * head_dim

        # GPT2 Conv1D weight shape is usually [D, 3D]
        weight = attn.c_attn.weight.data

        # Use V-projection slice for head importance
        v_weight = weight[:, 2 * D : 3 * D]

        norms = []
        for h in range(n_heads):
            start = h * head_dim
            end = (h + 1) * head_dim
            norms.append(v_weight[:, start:end].norm().item())

        result[layer_idx] = sorted(range(n_heads), key=lambda i: norms[i])[:n_prune]

    return result


def mask_gpt_heads(m, heads_to_prune):
    """
    Mask selected GPT2 attention heads by zeroing their Q/K/V and output slices.
    This is compatible with GPT2 versions that do not expose prune_heads().
    """
    for layer_idx, heads in heads_to_prune.items():
        attn = m.distilgpt.h[layer_idx].attn

        n_heads = attn.num_heads
        head_dim = attn.head_dim
        D = n_heads * head_dim

        with torch.no_grad():
            for h in heads:
                start = h * head_dim
                end = (h + 1) * head_dim

                # c_attn projects hidden -> Q, K, V together.
                # Weight shape: [D, 3D]
                attn.c_attn.weight[:, start:end] = 0
                attn.c_attn.weight[:, D + start : D + end] = 0
                attn.c_attn.weight[:, 2 * D + start : 2 * D + end] = 0

                if attn.c_attn.bias is not None:
                    attn.c_attn.bias[start:end] = 0
                    attn.c_attn.bias[D + start : D + end] = 0
                    attn.c_attn.bias[2 * D + start : 2 * D + end] = 0

                # c_proj projects concatenated heads back to hidden size.
                # Zero output contribution from this head.
                # GPT2 Conv1D weight shape: [D, D]
                attn.c_proj.weight[start:end, :] = 0

        print(f"Layer {layer_idx}: masked heads {heads}")


before = sum(p.numel() for p in model_pruned.parameters())

heads_to_prune = get_gpt_heads_to_prune(model_pruned, n_prune=4)

print("Heads to prune per GPT layer:")
for layer_idx, heads in heads_to_prune.items():
    print(f"  Layer {layer_idx}: {heads}")

mask_gpt_heads(model_pruned, heads_to_prune)

after = sum(p.numel() for p in model_pruned.parameters())

print(f"\nParams: {before:,} -> {after:,}")
print("Note: mask-based pruning keeps parameter count the same but removes selected head contributions.")

# Fine-tune after pruning
model_pruned = model_pruned.to(device)

opt_p = optim.AdamW(model_pruned.parameters(), lr=5e-5, weight_decay=1e-4)
crit_p = nn.CrossEntropyLoss(label_smoothing=0.05)

sched_p = optim.lr_scheduler.ReduceLROnPlateau(
    opt_p,
    mode="max",
    factor=0.5,
    patience=2
)

best_p_state, best_p_val = None, 0.0

for ep in range(10):
    model_pruned.train()
    ep_loss = 0.0

    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)

        opt_p.zero_grad()
        loss = crit_p(model_pruned(xb), yb)
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model_pruned.parameters(), 1.0)
        opt_p.step()

        ep_loss += loss.item()

    val_acc = evaluate(model_pruned, val_loader)
    sched_p.step(val_acc)

    if val_acc > best_p_val:
        best_p_val = val_acc
        best_p_state = copy.deepcopy(model_pruned.state_dict())

    print(
        f"Epoch {ep+1:02d}/10 | "
        f"loss={ep_loss/len(train_loader):.4f} | "
        f"val_acc={val_acc:.4f}"
    )

model_pruned.load_state_dict(best_p_state)
model_pruned = model_pruned.cpu().eval()

record_stage(
    "4 - GPT2 Head Masking / Structured Pruning",
    model_pruned,
    test_loader,
    disk_mb_val=model_disk_mb(model_pruned),
)

Heads to prune per GPT layer:
  Layer 0: [3, 5, 1, 4]
  Layer 1: [2, 9, 3, 8]
  Layer 2: [11, 3, 0, 1]
  Layer 3: [0, 8, 10, 5]
  Layer 4: [3, 0, 2, 8]
  Layer 5: [8, 10, 4, 0]
Layer 0: masked heads [3, 5, 1, 4]
Layer 1: masked heads [2, 9, 3, 8]
Layer 2: masked heads [11, 3, 0, 1]
Layer 3: masked heads [0, 8, 10, 5]
Layer 4: masked heads [3, 0, 2, 8]
Layer 5: masked heads [8, 10, 4, 0]

Params: 45,112,362 -> 45,112,362
Note: mask-based pruning keeps parameter count the same but removes selected head contributions.
Epoch 01/10 | loss=0.5279 | val_acc=0.9108
Epoch 02/10 | loss=0.4547 | val_acc=0.9238
Epoch 03/10 | loss=0.4305 | val_acc=0.9214
Epoch 04/10 | loss=0.4052 | val_acc=0.9535
Epoch 05/10 | loss=0.3948 | val_acc=0.9573
Epoch 06/10 | loss=0.3674 | val_acc=0.9521
Epoch 07/10 | loss=0.3630 | val_acc=0.9650
Epoch 08/10 | loss=0.3449 | val_acc=0.9631
Epoch 09/10 | loss=0.3409 | val_acc=0.9588
Epoch 10/10 | loss=0.3336 | val_acc=0.9621

[4 - GPT2 Head Masking / Structured Pruning]
  A

## Stage 5 — Early Exit Inference

`ee_single()` steps through the same DistilGPT2 blocks used during normal inference. Each exit head classifies from the final readout token at position `-1`.

Exit heads are retrained for 5 epochs with the backbone frozen. Threshold is chosen by maximising `early_exit_rate - 10 * accuracy_penalty`.


In [15]:
import copy
import numpy as np

# ── 5a. Define ee_single using GPT blocks step-by-step ────────────
@torch.no_grad()
def ee_single(m, x, threshold):
    """Single-sample early exit for DistilGPT2 sensor model."""
    hidden = m._embed(x)

    for i, block in enumerate(m.distilgpt.h):
        out = block(
            hidden,
            layer_past=None,
            attention_mask=None,
            head_mask=None,
            use_cache=False,
            output_attentions=False,
        )
        hidden = out[0]
        logits = m.exit_heads[i](hidden[:, -1])

        if F.softmax(logits, dim=-1).max().item() >= threshold:
            return logits, i   # early exit at layer i

    hidden = m.distilgpt.ln_f(hidden)
    readout_hidden = hidden[:, -1]
    cls_out = m.dropout(F.relu(m.pre_classifier(readout_hidden)))
    return m.classifier(cls_out), m.N_LAYERS   # full pass

@torch.no_grad()
def eval_ee(m, loader, threshold, max_samples=500):
    y_true, y_pred = [], []
    exit_dist = {i: 0 for i in range(m.N_LAYERS + 1)}
    for xb, yb in loader:
        for j in range(len(xb)):
            if len(y_true) >= max_samples: break
            logits, layer_idx = ee_single(m, xb[j:j+1], threshold)
            y_pred.append(logits.argmax(-1).item())
            y_true.append(yb[j].item())
            exit_dist[layer_idx] += 1
        if len(y_true) >= max_samples: break
    acc = accuracy_score(y_true, y_pred)
    early = sum(exit_dist[i] for i in range(m.N_LAYERS)) / len(y_true)
    avg_layer = sum(k * v for k, v in exit_dist.items()) / len(y_true)
    return acc, early, avg_layer, exit_dist

# ── 5b. Retrain exit heads on final model representations ─────────────
model_ee = copy.deepcopy(model).to(device)
for p in model_ee.parameters():
    p.requires_grad = False
for p in model_ee.exit_heads.parameters():
    p.requires_grad = True

exit_opt = optim.AdamW(model_ee.exit_heads.parameters(), lr=1e-3, weight_decay=1e-4)

for ep in range(5):
    model_ee.train()
    total_exit_loss = 0.0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        exit_opt.zero_grad()
        exits, _ = model_ee(xb, return_all_exits=True)
        loss = sum(criterion(e, yb) for e in exits) / len(exits)
        loss.backward()
        exit_opt.step()
        total_exit_loss += loss.item()
    print(f'Exit-head epoch {ep+1}/5 | loss={total_exit_loss/len(train_loader):.4f}')

model_ee = model_ee.cpu().eval()

# ── 5c. Choose threshold on validation subset ─────────────────────────
base_acc = results_table['1 - Baseline (FP32)']['Accuracy']
candidates = np.linspace(0.40, 0.95, 12)
best = None

for th in candidates:
    acc, early, avg_layer, dist = eval_ee(model_ee, val_loader, th, max_samples=500)
    penalty = max(0, base_acc - acc)
    score = early - 10 * penalty
    if best is None or score > best[0]:
        best = (score, th, acc, early, avg_layer, dist)

_, THRESH, val_acc_ee, early_rate, avg_layer, exit_dist = best
print(f'Chosen threshold: {THRESH:.2f}')
print(f'Val acc={val_acc_ee:.4f}, early_exit={early_rate*100:.1f}%, avg_layer={avg_layer:.2f}')
print('Exit distribution:', exit_dist)

# ── 5d. Record early-exit stage ───────────────────────────────────────


Exit-head epoch 1/5 | loss=0.6718
Exit-head epoch 2/5 | loss=0.6451
Exit-head epoch 3/5 | loss=0.6255
Exit-head epoch 4/5 | loss=0.6220
Exit-head epoch 5/5 | loss=0.6125
Chosen threshold: 0.90
Val acc=0.9380, early_exit=90.2%, avg_layer=2.01
Exit distribution: {0: 39, 1: 250, 2: 75, 3: 46, 4: 32, 5: 9, 6: 49}


TypeError: record_stage() got an unexpected keyword argument 'ram_fwd'

In [17]:
def ee_fwd(x):
    logits, _ = ee_single(model_ee, x, THRESH)
    return logits

record_stage(
    '5 - Early Exit',
    ee_fwd,
    test_loader,
    disk_mb_val=model_disk_mb(model_ee)
)


[5 - Early Exit]
  Accuracy  : 0.5889
  Latency   : 103.4 ms  (p95 = 215.9 ms)
  Peak RAM  : 0.00 MB
  Disk size : 172.13 MB


## Results — 5-Method Comparison


In [19]:
import pandas as pd

df = pd.DataFrame(results_table).T
df.index.name = 'Stage'
baseline_lat = df.loc['1 - Baseline (FP32)', 'Lat Mean (ms)']
df['Speedup vs FP32'] = (baseline_lat / df['Lat Mean (ms)']).round(2)

pd.set_option('display.float_format', '{:.2f}'.format)
pd.set_option('display.width', 130)

print('=' * 105)
print('  BREATHPRINT EDGE - DistilGPT2 Benchmark  (CPU, single-sample, 50 runs)')
print('=' * 105)
print(df.to_string())
print('=' * 105)
print()
print('Notes:')
print('  Latency  : mean of 50 single-sample CPU forward passes.')
print('  Speedup  : baseline_latency / method_latency  (>1.0 = faster than FP32).')
print('  Pruning  : GPT2 attention heads physically removed using built-in prune_heads().')
print('  On Pi 5  : absolute latency 5-15x higher; relative speedups remain valid.')


  BREATHPRINT EDGE - DistilGPT2 Benchmark  (CPU, single-sample, 50 runs)
                                            Accuracy  Lat Mean (ms)  Lat p95  (ms)  Peak RAM (MB)  Disk Size (MB)  Speedup vs FP32
Stage                                                                                                                             
1 - Baseline (FP32)                             0.94         170.80         211.20           0.01          172.13             1.00
2 - INT8 PTQ                                    0.93         169.70         215.20           0.01          167.04             1.01
3 - ONNX Runtime                                0.94         477.80         797.30           0.00          167.67             0.36
4 - GPT2 Head Masking / Structured Pruning      0.97         167.60         227.80           0.01          172.13             1.02
5 - Early Exit                                  0.59         103.40         215.90           0.00          172.13             1.65

Notes:
  

In [16]:
torch.save(copy.deepcopy(model).cpu().state_dict(),       'pi_fp32.pt')
torch.save(model_int8.state_dict(),                        'pi_int8.pt')
torch.save(model_pruned.state_dict(),                      'pi_pruned.pt')
torch.save(model_ee.state_dict(),                          'pi_ee.pt')
# distilgpt_sensor.onnx already saved in Stage 3
print('Model artefacts saved.')

Model artefacts saved.


## Raspberry Pi 5 Deployment

Saves all model artefacts and a self-contained inference script.
Download `pi_deployment.zip` from Colab file browser, then follow the printed steps.
